# Stage 9a — Joint CTC + Attention decoder on landmarks (Kaggle T4)

Same locked Stage 1 v2 encoder (4-layer Conformer, d=256, batch=32, 80 epochs, seed=42).  New variable: a 3-layer Transformer attention decoder + joint loss `L = 0.3·L_ctc + 0.7·L_attn`.

Decoder consumes the encoder's pre-upsample output [T=32, d=256] via cross-attention; CTC head continues on the post-upsample features.  Val CER is reported three ways: greedy CTC, greedy attention, and best-of-both per clip.

**Pass/fail** (per `configs/stage9.yaml`):
- Headline (full cohort, best-of-both): ≤ **0.59** (5 pts under Stage 1 v3 stripped 0.6383).
- Stretch: ≤ **0.53** — would warrant Stage 9b KenLM rescoring next.

**Wall-clock on Kaggle T4 (one session)**:

| Phase | Time |
|---|---|
| Skeleton cache build (if absent)        | ~30 min |
| 5-fold sweep (~50 min/fold, +decoder)   | ~4 h |
| Total                                   | ~4.5 h |

Fits comfortably in Kaggle's 9 h budget.  No external dataset required if the cache+manifest already live in your committed Task A / HRNet-swap / Stage 3 kernel — Cell 2 auto-locates.

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'mediapipe>=0.10.0' scipy --quiet

import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')

import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}  '
          f'({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')

## Cell 2 — Locate artifacts (cache, manifest, prior results)

In [ ]:
import os, glob, json, shutil

def _find(pattern: str):
    m = (glob.glob(f'/kaggle/working/**/{pattern}', recursive=True)
         + glob.glob(f'/kaggle/input/**/{pattern}',  recursive=True))
    return m[0] if m else None

CACHE_PATH        = _find('skeleton_features_t32.pt')
CV_MANIFEST_FOUND = _find('subject_cv5.json')
RESULTS_FOUND     = _find('stage9a_results.json')

OUT_CACHE    = CACHE_PATH        or '/kaggle/working/skeleton_features_t32.pt'
OUT_MANIFEST = CV_MANIFEST_FOUND or '/kaggle/working/subject_cv5.json'
RESULTS_PATH = '/kaggle/working/stage9a_results.json'
if RESULTS_FOUND and not os.path.exists(RESULTS_PATH):
    shutil.copy(RESULTS_FOUND, RESULTS_PATH)

CKPT_DIR = '/kaggle/working/checkpoints'
LOG_DIR  = '/kaggle/working/logs'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR,  exist_ok=True)

for label, path in [('cache',    OUT_CACHE),
                     ('manifest', OUT_MANIFEST),
                     ('results',  RESULTS_PATH)]:
    print(f'{label:<10s}: {path}  (exists={os.path.exists(path)})')

## Cell 3 — Config (locked Stage 1 v2 hyperparams + Stage 9 decoder knobs)

In [ ]:
import logging, random
import numpy as np

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s — %(message)s',
    handlers=[logging.StreamHandler(),
              logging.FileHandler(os.path.join(LOG_DIR, 'stage9a.log'))],
)

from wita_v2.configs.default import Config, DataConfig, EncoderConfig, TrainConfig

T_NATIVE     = 32
UPSAMPLE     = 2
D_MODEL      = 256
N_LAYERS     = 4
N_HEADS      = 4
CONV_KERNEL  = 15
DROPOUT      = 0.2
BATCH_SIZE   = 32
LR_PEAK      = 5e-4
WEIGHT_DECAY = 5e-2
GRAD_CLIP    = 1.0
NUM_EPOCHS   = 80
WARMUP_PCT   = 0.05
SEED         = 42
FOLDS        = list(range(5))

# Stage-9-specific:
DEC_N_LAYERS = 3
DEC_N_HEADS  = 4
LAMBDA_CTC   = 0.3
VARIANT_NAME = 'stage9a'

cfg = Config(
    data=DataConfig(hf_repo_id='yewon816/WiTA', lang='english',
                    max_zips=None, max_frames=64, seed=SEED),
    encoder=EncoderConfig(arch='siglip'),
    train=TrainConfig(num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
                      lr=LR_PEAK, weight_decay=WEIGHT_DECAY,
                      grad_clip=GRAD_CLIP, num_workers=2,
                      warmup_pct=WARMUP_PCT, seed=SEED,
                      checkpoint_dir=CKPT_DIR),
).build()
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
print(f'Device         : {cfg.device}')
print(f'lambda_ctc     : {LAMBDA_CTC}')
print(f'decoder layers : {DEC_N_LAYERS}  heads : {DEC_N_HEADS}')

## Cell 4 — Build skeleton cache + CV manifest if missing

Auto-streams + extracts MediaPipe over the 39 English-subset signers (~30 min on T4).  No-op if both files exist.

In [ ]:
if not os.path.exists(OUT_CACHE):
    from wita_v2.datasets.subject_splits import stream_and_index_with_subjects
    from wita_v2.datasets.skeleton_cache  import extract_skeleton_features
    print('Building skeleton cache from scratch...')
    samples = stream_and_index_with_subjects(cfg)
    extract_skeleton_features(
        samples=samples, out_path=OUT_CACHE,
        T_native=T_NATIVE, dtype=torch.float16,
    )
cache = torch.load(OUT_CACHE, map_location='cpu', weights_only=False)
print(f'cache: {len(cache["feats"])} clips, '
      f'detect_rate={cache.get("frame_detect_rate",0)*100:.1f}%')

if not os.path.exists(OUT_MANIFEST):
    from wita_v2.datasets.cv_splits import build_cv5_manifest, save_cv5_manifest
    print('Building CV manifest from scratch...')
    fake = [(b'', cache['labels'][i], cache['subjects'][i])
            for i in range(len(cache['feats']))]
    manifest = build_cv5_manifest(fake, n_folds=5, seed=SEED)
    save_cv5_manifest(manifest, OUT_MANIFEST)
else:
    from wita_v2.datasets.cv_splits import load_cv5_manifest
    manifest = load_cv5_manifest(OUT_MANIFEST)
print(f'manifest: {manifest["n_subjects_total"]} signers, {manifest["n_folds"]} folds')

## Cell 5 — Run the 5-fold sweep (joint CTC + attention)

Each fold trains the encoder + decoder jointly for 80 epochs.  Results are written incrementally so a Kaggle disconnect is recoverable: re-run this cell to skip completed folds.

In [ ]:
from wita_v2.training.stage9_train   import train_one_fold
from wita_v2.datasets.cv_splits       import fold_indices
from wita_v2.datasets.skeleton_augment import LandmarkAugment

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        all_results = json.load(f)
    completed = {(r['fold'], r['variant']) for r in all_results}
    print(f'Resuming — {len(completed)} folds already complete.')
else:
    all_results = []
    completed = set()

train_aug = LandmarkAugment()    # Stage 1 v2 defaults (spatial + temporal)

for fold in FOLDS:
    if (fold, VARIANT_NAME) in completed:
        continue
    train_idx, val_idx = fold_indices(manifest, fold, cache['subjects'])
    result = train_one_fold(
        cache=cache, train_idx=train_idx, val_idx=val_idx, cfg=cfg,
        fold=fold, variant=VARIANT_NAME,
        num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr_peak=LR_PEAK,
        weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP, dropout=DROPOUT,
        d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
        conv_kernel=CONV_KERNEL, upsample=UPSAMPLE, warmup_pct=WARMUP_PCT,
        dec_n_layers=DEC_N_LAYERS, dec_n_heads=DEC_N_HEADS,
        lambda_ctc=LAMBDA_CTC,
        transform=train_aug,
        checkpoint_dir=CKPT_DIR, log_dir=LOG_DIR,
    )
    summary = {k: v for k, v in result.items() if k != 'history'}
    all_results.append(summary)
    with open(RESULTS_PATH, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'  saved → {RESULTS_PATH}\n')
print(f'\nAll {len(all_results)}/{len(FOLDS)} folds done.')

## Cell 6 — Aggregate (dual-cohort) + verdict

In [ ]:
import numpy as np
from wita_v2.reports.template.stripped_cohort import dual_cohort_summary

with open(RESULTS_PATH) as f:
    all_results = json.load(f)
per_fold = {r['fold']: r for r in all_results if r['variant'] == VARIANT_NAME}
best = [per_fold[f]['best_val_cer']  for f in FOLDS if f in per_fold]
ctc_nlls = [per_fold[f]['final_train_ctc_nll'] for f in FOLDS if f in per_fold]
att_nlls = [per_fold[f]['final_train_attn_nll'] for f in FOLDS if f in per_fold]

print(f'Stage 9a — {len(best)} folds complete\n')
print(' fold    best CER     final ctc nll    final attn nll')
for f in FOLDS:
    if f not in per_fold:
        print(f'  {f:>2d}    {"-":>10}      {"-":>10}        {"-":>10}'); continue
    r = per_fold[f]
    print(f'  {f:>2d}    {r["best_val_cer"]:.4f}        {r["final_train_ctc_nll"]:.4f}            {r["final_train_attn_nll"]:.4f}')

if len(best) >= 2:
    s = dual_cohort_summary(RESULTS_PATH, VARIANT_NAME)
    print(f'\n     full cohort   : {s["full_mean"]:.4f} ± {s["full_std"]:.4f}')
    print(f'     PHW/KIM-strip : {s["stripped_mean"]:.4f} ± {s["stripped_std"]:.4f}')
    print(f'     Δ (full-strip): {s["delta_mean"]:+.4f}')

    print('\n=== Stage 9a verdict ===')
    full = s['full_mean']
    STAGE1V3_STRIPPED = 0.6383
    if full <= 0.53:
        print(f'  ✅ STRETCH ({full:.4f} ≤ 0.53) — Stage 9b KenLM rescoring is warranted.')
    elif full <= 0.59:
        print(f'  ✅ HEADLINE ({full:.4f} ≤ 0.59) — joint CTC+attention helps. Consider 9b.')
    elif full < STAGE1V3_STRIPPED - 0.01:
        print(f'  ⚖  MARGINAL ({full:.4f} < {STAGE1V3_STRIPPED-0.01:.4f}). Attention helps a little; 9b unlikely to bridge to stretch.')
    elif abs(full - STAGE1V3_STRIPPED) <= 0.01:
        print(f'  ⚖  TIE with Stage 1 v3 stripped ({full:.4f} ≈ {STAGE1V3_STRIPPED:.4f}).  Attention adds nothing; do not pursue 9b.')
    else:
        print(f'  ❌ REGRESS ({full:.4f} > {STAGE1V3_STRIPPED:.4f}). Attention decoder destabilises training; tune lambda_ctc or shrink decoder.')

## Cell 7 — Per-signer val CER scatter (dual-cohort colored)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

per_signer = {}
for r in all_results:
    if r['variant'] != VARIANT_NAME: continue
    per_signer.update(r.get('best_per_signer_val_cer', {}) or {})
items = sorted(per_signer.items(), key=lambda kv: kv[1])
xs = list(range(len(items)))
ys = [v for _, v in items]
DATASET_LIMIT = ['PHW', 'KIM']
MODEL_HARD    = ['PJH','SYB','KJM','KNY','LKS','YMG']
def _c(s):
    if s in DATASET_LIMIT: return '#7f7f7f'
    if s in MODEL_HARD:    return '#d62728'
    return '#1f77b4'
colors = [_c(s) for s,_ in items]
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.scatter(xs, ys, s=30, c=colors)
ax.axhline(0.55, color='green', linestyle='--', alpha=0.5, label='easy ≤ 0.55')
ax.axhline(0.75, color='red',   linestyle='--', alpha=0.5, label='hard ≥ 0.75')
ax.set_xticks(xs); ax.set_xticklabels([s for s,_ in items], rotation=90, fontsize=7)
ax.set_ylabel('val CER (best of CTC / attn)'); ax.set_xlabel('signer (sorted by CER)')
ax.set_title(f'Stage 9a — per-signer val CER  '
             '[grey=PHW/KIM dataset-side, red=model-side hard]')
ax.legend(frameon=False); ax.grid(True, linestyle=':', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'stage9a_per_signer_scatter.png'), dpi=140)
plt.show()

## Cell 8 — Commit kernel

Save & Run All so these survive the session:
- `/kaggle/working/stage9a_results.json`            — per-fold incremental results
- `/kaggle/working/checkpoints/stage9a_fold*_best.pt` — encoder + decoder weights
- `/kaggle/working/logs/stage9a_per_signer_scatter.png`

Stage 9b (KenLM rescoring) will attach this kernel's checkpoints and apply prefix-beam-search without retraining.